# UCI Sticky-Sampler Sparsity Ablation — PIW Sweep

Sticky Boomerang vs. Sticky ZigZag on Boston, swept over the spike-and-slab prior inclusion
weight $w$. Loads the raw skeleton/resample output of `uci_sparsity_ablation.py` and produces:
a metrics table, an event-type breakdown table, and three figures (accuracy/sparsity/noise
tradeoff, posterior noise vs. $w$, refresh-reach vs. posterior spread).


In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import Tensor
from torch.distributions import Normal

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 11,
})

import os
if Path.cwd().name == "notebooks":
    os.chdir("..")


## 1. Load runs and predictions

Discovers every `piw_*/` skeleton file under one `(dataset, split, hidden_variant)`, then
computes the per-draw predictive mean $f_s(x) = \mathrm{NN}_{\beta_s}(x)$ and noise
$\sigma_s = \exp(\log\sigma_s)$ for each of the $S$ resampled posterior draws
$\beta_s = (\text{weights}_s, \log\sigma_s)$.


In [ ]:
from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)
from sazz.gpu_friendly.scripts.uci_sparsity_ablation import build_bm_only

DATASET, SPLIT_ID, HIDDEN_VARIANT = "boston", 0, "small"
OUT_MAPS_DIR = Path("results/maps/uci_sparsity_ablation")
OUT_DIR = Path("results/paper/uci_sparsity_ablation")
SPLIT_ROOT = OUT_DIR / DATASET / f"split_{SPLIT_ID:02d}"
MAP_CKPT_PATH = OUT_MAPS_DIR / f"{DATASET}_split{SPLIT_ID:02d}_map.pt"

LABELS = {"grid_sticky_zigzag": "Sticky ZigZag", "grid_sticky_boomerang": "Sticky Boomerang"}
COLORS = {"grid_sticky_zigzag": "#1F77B4", "grid_sticky_boomerang": "#FA5C00"}

# {w: {stem: path}} for every piw_<w>/ dir holding at least one *_skeleton.pt
piw_dirs: dict[float, dict[str, Path]] = {}
for p in sorted(SPLIT_ROOT.glob("piw_*")):
    files = list(p.glob("*_skeleton.pt"))
    if not files:
        continue
    w = float(p.name.removeprefix("piw_"))
    piw_dirs[w] = {f.stem.removesuffix("_skeleton"): f for f in files}

assert piw_dirs, f"No skeleton .pt files found under {SPLIT_ROOT}. Run uci_sparsity_ablation.py first."
map_ckpt = torch.load(MAP_CKPT_PATH, map_location="cpu", weights_only=False)

# runs: {(w, stem): {payload, diag_df, preds, mean_pred, noise_samples, weight_samples, ...}}
runs: dict[tuple[float, str], dict] = {}
for w, stems in sorted(piw_dirs.items()):
    for stem, f in stems.items():
        payload = torch.load(f, map_location="cpu", weights_only=False)
        diag_log = payload.get("diagnostics")
        runs[(w, stem)] = dict(payload=payload, diag_df=pd.DataFrame(diag_log) if diag_log else None)

cfg = BNNConfig(layer_sizes=map_ckpt["layer_sizes"], activation=map_ckpt["activation"],
                prior_sigma_scale=map_ckpt["prior_sigma_scale"])
raw = load_raw_datasets((DATASET,))
data = make_split(*raw[DATASET], seed=BASE_SEED + SPLIT_ID, dtype=DTYPE, device=DEVICE)
X_test, y_test, y_std = data["X_test"], data["y_test"], data["y_std"]
bm = build_bm_only(data, cfg)
D = bm.D

@torch.no_grad()
def predict_all(weight_samples: Tensor, X_new: Tensor) -> Tensor:
    """f_s(X_new) for every posterior draw s -- [S, N]."""
    return torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_new,)).squeeze(-1)
        for beta in weight_samples
    ])

for (w, stem), r in runs.items():
    samples = r["payload"]["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
    r["weight_samples"] = samples[:, :-1]
    r["preds"] = predict_all(r["weight_samples"], X_test)          # [S, N]
    r["mean_pred"] = r["preds"].mean(0)
    r["epist_std"] = r["preds"].std(0)
    r["noise_samples"] = samples[:, -1].exp()                      # [S]
    r["noise_std_eff"] = float(r["noise_samples"].mean())
    r["total_std"] = (r["epist_std"] ** 2 + r["noise_std_eff"] ** 2).sqrt()

print(f"{DATASET} split_{SPLIT_ID:02d} {HIDDEN_VARIANT} | D={D} | "
      f"{len(runs)} runs across {len(piw_dirs)} PIW values: {sorted(piw_dirs)}")


## 2. Metrics table

Posterior predictive is the $S$-draw Gaussian mixture $p(y\mid x) = \frac1S\sum_s
\mathcal N(y; f_s(x), \sigma_s^2)$. Reported per $(w, \text{sampler})$:

$$\mathrm{RMSE} = \sqrt{\textstyle\frac1N\sum_n (\bar f(x_n) - y_n)^2}\,\hat\sigma_y,
\qquad \bar f = \frac1S\sum_s f_s$$
$$\mathrm{NLL} = -\frac1N\sum_n \log\Big(\frac1S\sum_s \mathcal N(y_n; f_s(x_n), \sigma_s^2)\Big)
+ \log\hat\sigma_y$$

CRPS is the closed-form Gaussian-mixture CRPS (Gneiting & Raftery 2007), evaluated exactly
for pairs of mixture components. Coverage uses the two-moment Gaussian approximation to the
mixture, $\mathcal N(\bar f, \,\mathrm{Var}_s[f_s] + \overline{\sigma^2})$.


In [ ]:
CRPS_MIX_SUBSAMPLE = 256  # CRPS is O(S^2) -- subsample draws for speed

def _crps_gauss_term(m: Tensor, s: Tensor) -> Tensor:
    """E|X - m| for X ~ N(0, s^2)."""
    d = Normal(0.0, 1.0)
    z = m / s
    return m * (2 * d.cdf(z) - 1) + 2 * s * d.log_prob(z).exp()

def compute_rmse(mean_pred): return float(((mean_pred - y_test) ** 2).mean().sqrt()) * y_std

def compute_nll_mixture(preds, noise):
    lp = Normal(preds, noise[:, None]).log_prob(y_test)
    return float(-(torch.logsumexp(lp, 0) - math.log(preds.shape[0])).mean() + math.log(y_std))

def compute_crps_mixture(preds, noise, n_sub=CRPS_MIX_SUBSAMPLE, seed=0):
    idx = torch.randperm(preds.shape[0], generator=torch.Generator().manual_seed(seed))[:n_sub]
    mu, sd = preds[idx] * y_std, (noise[idx] * y_std)[:, None].expand(-1, preds.shape[1])
    yt = y_test * y_std
    term1 = _crps_gauss_term(yt[None] - mu, sd).mean(0)
    diff = mu[:, None] - mu[None, :]
    s2 = (sd[:, None] ** 2 + sd[None, :] ** 2).sqrt()
    term2 = _crps_gauss_term(diff, s2).mean((0, 1))
    return float((term1 - 0.5 * term2).mean())

def compute_coverage(mean_pred, total_std, level):
    z = Normal(0.0, 1.0).icdf(torch.tensor(0.5 + level / 2))
    return float(((y_test - mean_pred).abs() <= z * total_std).float().mean())

rows = []
for (w, stem), r in runs.items():
    pay = r["payload"]
    rows.append({
        "PIW": w, "Sampler": LABELS.get(stem, stem),
        "RMSE": compute_rmse(r["mean_pred"]),
        "NLL": compute_nll_mixture(r["preds"], r["noise_samples"]),
        "CRPS": compute_crps_mixture(r["preds"], r["noise_samples"]),
        "Cov 90%": compute_coverage(r["mean_pred"], r["total_std"], 0.90),
        "Cov 95%": compute_coverage(r["mean_pred"], r["total_std"], 0.95),
        "Achieved sparsity": float(pay["frozen_mask_final"].float().mean()),
    })
metrics_df = pd.DataFrame(rows).sort_values(["PIW", "Sampler"]).set_index(["PIW", "Sampler"])

def _highlight_coverage(col):
    target = 0.90 if "90" in col.name else 0.95
    best = (col - target).abs().idxmin()
    return ["background-color: #68dc0f" if idx == best else "" for idx in col.index]

metrics_df.style \
    .highlight_min(subset=["RMSE", "NLL", "CRPS"], color="#68dc0f") \
    .apply(_highlight_coverage, subset=["Cov 90%", "Cov 95%"]) \
    .format(precision=4)


## 3. Event-type breakdown

Every sampler-loop iteration ends in exactly one event: **bounce** (Poisson-thinning
candidate accepted, velocity reflects), **no_event** (thinning rejected the whole grid
window), **freeze**/**thaw** (a coordinate hits/leaves zero, sticky-only), or **refresh**
(velocity fully resampled, Boomerang-only). Table entries are each event type's share of
total loop iterations, $(w, \text{sampler})$-wise.


In [ ]:
EVENT_TYPES = ["bounce", "no_event", "freeze", "thaw", "refresh"]

event_rows = []
for (w, stem), r in runs.items():
    if r["diag_df"] is None:
        continue
    row = {"PIW": w, "Sampler": LABELS.get(stem, stem)}
    row.update({e: (r["diag_df"]["event_type"] == e).mean() for e in EVENT_TYPES})
    event_rows.append(row)

event_df = pd.DataFrame(event_rows).sort_values(["PIW", "Sampler"]).set_index(["PIW", "Sampler"])
event_df.style.format(precision=4).background_gradient(cmap="Blues", axis=None)


## 4. Figure 1 — accuracy vs. sparsity, achieved sparsity, noise

**(a)** achieved sparsity $\frac1D\sum_i \mathbb 1[\hat\beta_i = 0]$ (final frozen mask)
against the nominal prior weight $w$. **(b)** RMSE against *achieved* sparsity (not $w$):
band is $\mathrm{mean}\pm\mathrm{std}$ of the **per-draw** RMSE
$\sqrt{\frac1N\sum_n(f_s(x_n)-y_n)^2}\,\hat\sigma_y$ across the $S$ draws — the spread of
individual-draw accuracy, not a CI on the point estimate. **(c)** posterior noise
$\sigma_s=\exp(\log\sigma_s)$ against $w$: median line, IQR band.


In [ ]:
fig, (ax_sp, ax_acc, ax_noise) = plt.subplots(1, 3, figsize=(15, 4.3))
stems = list(LABELS.keys())

for stem in stems:
    color = COLORS[stem]
    keys = sorted(k for k in runs if k[1] == stem)
    if not keys:
        continue
    ws = [w for w, _ in keys]
    sp = [float(runs[k]["payload"]["frozen_mask_final"].float().mean()) for k in keys]
    ax_sp.plot(ws, sp, marker="o", ms=5, color=color, label=LABELS[stem])

    # (b) RMSE vs. achieved sparsity, band = std of per-draw RMSE
    pts = []
    for k in keys:
        r = runs[k]
        achieved = float(r["payload"]["frozen_mask_final"].float().mean())
        per_draw_rmse = ((r["preds"] - y_test[None]) ** 2).mean(1).sqrt() * y_std
        pts.append((achieved, float(per_draw_rmse.mean()), float(per_draw_rmse.std())))
    pts.sort()
    sp_x, mu, sd = map(np.array, zip(*pts))
    ax_acc.plot(sp_x, mu, marker="o", ms=5, color=color, label=LABELS[stem])
    ax_acc.fill_between(sp_x, mu - sd, mu + sd, color=color, alpha=0.18, linewidth=0)

    # (c) noise vs. w, median + IQR
    noise_pts = [(w, np.median(runs[k]["noise_samples"].numpy()),
                  *np.percentile(runs[k]["noise_samples"].numpy(), [25, 75])) for w, k in zip(ws, keys)]
    ws2, med, q25, q75 = map(np.array, zip(*noise_pts))
    ax_noise.plot(ws2, med, marker="o", ms=5, color=color, label=LABELS[stem])
    ax_noise.fill_between(ws2, q25, q75, color=color, alpha=0.18, linewidth=0)

ax_sp.plot([0, 1], [1, 0], color="grey", ls=":", lw=1, label="y = 1 - w")
ax_sp.set(xlabel="Nominal $w$", ylabel="Achieved sparsity", title="(a) Achieved vs. nominal sparsity",
          xlim=(0, 1), ylim=(0, 1))
ax_acc.set(xlabel="Achieved sparsity", ylabel="RMSE", title="(b) Accuracy vs. sparsity", xlim=(0, 1))
ax_noise.set(xlabel="Nominal $w$", ylabel=r"Posterior $\sigma$", title="(c) Noise vs. $w$", xlim=(0, 1))
for ax in (ax_sp, ax_acc, ax_noise):
    ax.legend(fontsize=7)

fig.suptitle(f"{DATASET.capitalize()} — sparsity, accuracy, noise across the PIW sweep", y=1.02)
plt.tight_layout()
plt.savefig(f"sparsity_ablation_{DATASET}.pdf", bbox_inches="tight", dpi=200)
plt.show()


## 5. Figure 2 — posterior noise distribution

Full posterior draws $\sigma_s = \exp(\log\sigma_s)$, $s=1,\dots,S$, per $(w,\text{sampler})$
— same quantity as Figure 1(c), unsummarized.


In [ ]:
piw_values = sorted(set(w for w, _ in runs))
violin_width = 0.8 / len(stems)

fig, ax = plt.subplots(figsize=(9, 5))
for i, stem in enumerate(stems):
    keys = [(w, stem) for w in piw_values if (w, stem) in runs]
    if not keys:
        continue
    offset = (i - (len(stems) - 1) / 2) * violin_width
    positions = [piw_values.index(w) + offset for w, _ in keys]
    data = [runs[k]["noise_samples"].numpy() for k in keys]
    parts = ax.violinplot(data, positions=positions, widths=violin_width * 0.9,
                           showmedians=True, showextrema=False)
    for body in parts["bodies"]:
        body.set_facecolor(COLORS[stem]); body.set_edgecolor(COLORS[stem]); body.set_alpha(0.55)
    parts["cmedians"].set_color(COLORS[stem])
    ax.plot([], [], color=COLORS[stem], lw=3, label=LABELS[stem])  # legend proxy

ax.set_xticks(range(len(piw_values)))
ax.set_xticklabels([f"{w:g}" for w in piw_values])
ax.set(xlabel="Nominal $w$", ylabel=r"Posterior $\sigma$ draws",
       title=f"{DATASET.capitalize()} — posterior noise distribution vs. $w$")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 6. Figure 3 — refresh reach vs. posterior spread (Boomerang only)

Boomerang bounces on excess curvature $\langle v,\, \nabla U(x) - \Sigma^{-1}(x-x_{\mathrm{ref}})
\rangle_+$, identically 0 if the reference exactly matched the target. Few bounces is
ambiguous: a good reference needs little correcting, *or* the sampler is stuck. Refreshment
draws $v_i \sim \mathcal N(0,\Sigma_i)$, so its per-coordinate reach is $\sqrt{\Sigma_i} =
\Sigma^{-1}_i{}^{-1/2}$. Plotted against each active coordinate's resampled posterior std, at
the sparsest $w$ (refresh matters most where sticky dynamics dominate): points **above**
$y=x$ have posterior spread refresh cannot reach in one draw — a sign the sampler
under-explores that coordinate, not that the reference is simply well matched.


In [ ]:
BOOM_STEM = "grid_sticky_boomerang"
boom_keys = sorted(k for k in runs if k[1] == BOOM_STEM)
assert boom_keys, f"no {BOOM_STEM} runs found"


refresh_reach = (1.0 / map_ckpt["Sigma_inv"].to(dtype=DTYPE))[:-1].sqrt()

ratios = {}
log_ratios = {}       # w -> log(post_std / refresh_reach) for active coords
frac_above = {}        # w -> fraction of active coords exceeding refresh reach
for w, stem in boom_keys:
    r = runs[(w, stem)]
    post_std = r["weight_samples"].std(dim=0)
    active = ~r["payload"]["frozen_mask_final"][:-1]
    ratio = (post_std[active] / refresh_reach[active])
    ratios[w] = ratio
    log_ratios[w] = torch.log(ratio).numpy()
    frac_above[w] = float((ratio > 1).float().mean())

ws = sorted(log_ratios)
fig, ax = plt.subplots(figsize=(8, 4.5))
# parts = ax.violinplot([log_ratios[w] for w in ws], positions=range(len(ws)),
#                        showmedians=True, showextrema=False)
parts = ax.violinplot([ratios[w] for w in ws], positions=range(len(ws)),
                       showmedians=True, showextrema=False)
for body in parts["bodies"]:
    body.set_facecolor(COLORS[BOOM_STEM]); body.set_edgecolor(COLORS[BOOM_STEM]); body.set_alpha(0.55)
parts["cmedians"].set_color(COLORS[BOOM_STEM])
ax.axhline(1.0, color="black", ls="--", lw=1.0, label="post_std = refresh reach")

ax.set_xticks(range(len(ws)))
ax.set_xticklabels([f"{w:g}" for w in ws])
ax.set(xlabel="Nominal $w$", ylabel=r"posterior std / refresh reach",
       title=f"{DATASET.capitalize()}, Sticky Boomerang — refresh reach vs. posterior spread, all $w$")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for w in ws:
    print(f"[w={w:g}] {frac_above[w]:.1%} of active coordinates exceed refresh reach "
          f"(n={len(log_ratios[w])})")


In [ ]:
# --- Posterior inclusion probability for one (PIW, sampler), sliced onto the network ---
import matplotlib as mpl

STEM     = "grid_sticky_zigzag"   # "grid_sticky_zigzag" or "grid_sticky_boomerang"
PIW_VIS  = 0.01                     # which point of the sweep to visualise
ZERO_TOL = 0.0                     # exact-zero test; use e.g. 1e-8 for near-zero

assert (PIW_VIS, STEM) in runs, f"no run for (w={PIW_VIS}, {STEM}); have {sorted(runs)}"
r = runs[(PIW_VIS, STEM)]

# weight_samples is [S, D-1] -- network weights + biases only, log_sigma already dropped
ws_arr    = r["weight_samples"].numpy()
S, Dw     = ws_arr.shape
incl_prob = (np.abs(ws_arr) > ZERO_TOL).mean(axis=0)          # P(active), one per network coord

layer_sizes = map_ckpt["layer_sizes"]                          # e.g. [13, 50, 1]
activation  = map_ckpt["activation"]
n_net = sum(n_in * n_out + n_out for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]))
assert n_net == Dw, (n_net, Dw)                                # weights+biases, no log_sigma

# Walk named_parameters() order: layers.i.weight [n_out, n_in], then layers.i.bias [n_out]
W_incl, b_incl, off = [], [], 0
for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
    W_incl.append(incl_prob[off:off + n_out * n_in].reshape(n_out, n_in)); off += n_out * n_in
    b_incl.append(incl_prob[off:off + n_out]);                            off += n_out
assert off == Dw, (off, Dw)

print(f"{LABELS[STEM]}  w={PIW_VIS:g}   (S={S} draws)")
for li, (Wp, bp) in enumerate(zip(W_incl, b_incl)):
    print(f"  layer {li}: W{Wp.shape}  mean incl={Wp.mean():.3f}   "
          f"bias mean incl={bp.mean():.3f}   fully-excluded weights={(Wp == 0).mean():.3f}")


In [ ]:
# --- Draw the network, edges coloured by posterior inclusion probability ---
cmap = mpl.colormaps["RdYlGn"]                  # 0 -> red (excluded), 1 -> green (included)
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)

xs = np.arange(len(layer_sizes))
def _ys(n):
    return np.linspace(0, 1, n) if n > 1 else np.array([0.5])
node_y = [_ys(n) for n in layer_sizes]

fig, ax = plt.subplots(figsize=(4 + 1.6 * len(layer_sizes), 9))

# edges: weight[j, k] connects node k in layer li to node j in layer li+1
for li, Wp in enumerate(W_incl):
    n_out, n_in = Wp.shape
    y0, y1 = node_y[li], node_y[li + 1]
    order = np.argsort(Wp.ravel())             # draw excluded (red) first, included (green) on top
    for idx in order:
        j, k = divmod(idx, n_in)
        p = Wp[j, k]
        ax.plot([xs[li], xs[li + 1]], [y0[k], y1[j]],
                color=cmap(norm(p)), lw=0.4 + 2.2 * p,
                alpha=0.15 + 0.85 * p, zorder=1, solid_capstyle="round")

# nodes coloured by mean incoming-weight inclusion (input layer: neutral grey)
for li, n in enumerate(layer_sizes):
    node_c = ["0.6"] * n if li == 0 else [cmap(norm(W_incl[li - 1][j].mean())) for j in range(n)]
    ax.scatter(np.full(n, xs[li]), node_y[li], s=260, c=node_c,
               edgecolors="black", linewidths=0.8, zorder=3)

labels = ([f"input\n({layer_sizes[0]})"]
          + [f"hidden {i}\n({s}, {activation})" for i, s in enumerate(layer_sizes[1:-1], 1)]
          + [f"output\n({layer_sizes[-1]})"])
for x, lab in zip(xs, labels):
    ax.text(x, -0.08, lab, ha="center", va="top", fontsize=11)

ax.set_xlim(xs[0] - 0.3, xs[-1] + 0.3); ax.set_ylim(-0.15, 1.05); ax.axis("off")
ax.set_title(f"{LABELS[STEM]}, {DATASET.capitalize()}  (w = {PIW_VIS:g}) — "
             f"posterior inclusion probability per weight\n"
             f"(green = almost always active, red = almost always pruned; "
             f"line weight = certainty)", fontsize=12)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax,
             fraction=0.03, pad=0.02, label="P(weight active)")
plt.tight_layout()
plt.show()
# fig.savefig(f"network_inclusion_{DATASET}_{STEM}_w{PIW_VIS:g}.pdf", bbox_inches="tight")


In [ ]:
# --- Keep only weights active > cutoff, re-evaluate on the test set (same (PIW, sampler)) ---
INCL_CUTOFF = 0.90

r          = runs[(PIW_VIS, STEM)]
samples    = r["payload"]["samples"].to(dtype=DTYPE)          # [S, D]  (last col = log_sigma)
weight_s   = r["weight_samples"]                              # [S, D-1]
keep_mask  = torch.as_tensor(incl_prob > INCL_CUTOFF, dtype=weight_s.dtype)   # [D-1]

weight_masked = weight_s * keep_mask                         # zero the <=cutoff coords in every draw

preds_m   = predict_all(weight_masked, X_test)               # [S, N]   (ablation predict_all: no bm arg)
mean_m    = preds_m.mean(0)
noise_m   = samples[:, -1].exp()                             # [S]  (log_sigma untouched by the mask)
total_m   = (preds_m.std(0) ** 2 + float(noise_m.mean()) ** 2).sqrt()

n_kept = int(keep_mask.sum())
print(f"{LABELS[STEM]}  w={PIW_VIS:g}: kept {n_kept} / {weight_s.shape[1]} weights "
      f"active >{INCL_CUTOFF:.0%} of the time ({n_kept / weight_s.shape[1]:.1%})\n")
print(f"  RMSE    : {compute_rmse(mean_m):.4f}   "
      f"(full posterior: {compute_rmse(r['mean_pred']):.4f})")
print(f"  NLL     : {compute_nll_mixture(preds_m, noise_m):.4f}   "
      f"(full posterior: {compute_nll_mixture(r['preds'], r['noise_samples']):.4f})")
print(f"  CRPS    : {compute_crps_mixture(preds_m, noise_m):.4f}   "
      f"(full posterior: {compute_crps_mixture(r['preds'], r['noise_samples']):.4f})")
print(f"  Cov 90% : {compute_coverage(mean_m, total_m, 0.90):.3f}   "
      f"(full: {compute_coverage(r['mean_pred'], r['total_std'], 0.90):.3f})")
print(f"  Cov 95% : {compute_coverage(mean_m, total_m, 0.95):.3f}   "
      f"(full: {compute_coverage(r['mean_pred'], r['total_std'], 0.95):.3f})")
